# Explore ingestion and alpha factors

Scratch notebook for pulling data through `src/data/ingest.py` and trying out the factor functions in `src/data/features.py` interactively. Nothing in here is part of the pipeline itself — it's just a place to poke at the functions and sanity-check the numbers.

Kernel: select **portfolio-constructor** (the project's `.venv`).

In [ ]:
import sys
from pathlib import Path

# Notebooks run with the notebook's own directory as the working directory,
# so the project root (parent of notebooks/) needs to be on sys.path for
# `from src import ...` imports to resolve.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data import ingest, features

## 1. Pull data (cached after the first run)

In [ ]:
tickers = config.TICKER_UNIVERSE + [config.BENCHMARK_TICKER]

prices_long = ingest.load_or_fetch_prices(tickers, config.DATA_START_DATE, config.DATA_END_DATE)
dividends = ingest.load_or_fetch_dividends(config.TICKER_UNIVERSE, config.DATA_START_DATE, config.DATA_END_DATE)

prices = ingest.to_wide_adj_close(prices_long)

print("prices_long:", prices_long.shape)
print("dividends:", dividends.shape)
print("prices (wide):", prices.shape)
prices.tail()

In [ ]:
# Quick visual sanity check: do these look like real adjusted-close price series?
prices[["AAPL", "JPM", config.BENCHMARK_TICKER]].plot(figsize=(10, 4), title="Adjusted close")
plt.show()

## 2. Try the factor functions as of a given date

Each factor only uses price/dividend history strictly before `as_of_date` — change the date below and rerun to see the rankings shift.

In [ ]:
as_of_date = pd.Timestamp("2023-01-03")

momentum = features.compute_momentum(prices, as_of_date)
low_volatility = features.compute_low_volatility(prices, as_of_date)
dividend_yield = features.compute_dividend_yield(dividends, prices, as_of_date)

factor_table = pd.DataFrame(
    {"momentum": momentum, "low_volatility": low_volatility, "dividend_yield": dividend_yield}
).drop(index=config.BENCHMARK_TICKER, errors="ignore")

factor_table.sort_values("momentum", ascending=False)

In [ ]:
print("Top 5 momentum:")
print(factor_table["momentum"].sort_values(ascending=False).head())

print("\nTop 5 lowest volatility:")
print(factor_table["low_volatility"].sort_values().head())

print("\nTop 5 dividend yield:")
print(factor_table["dividend_yield"].sort_values(ascending=False).head())

## 3. Rank stocks and optimise a portfolio

Combine the three factors into one score, take the top `TOP_N_HOLDINGS`, then find mean-variance weights (Ledoit-Wolf shrinkage, max Sharpe) using the trailing `TRAIN_WINDOW_YEARS` of price history before `as_of_date` — the same window the real walk-forward backtest will use at each rebalance.

In [ ]:
from src.models import ranker
from src.portfolio import optimise

# Use factor_table, not the raw momentum/low_volatility/dividend_yield series
# directly — factor_table already excludes the SPY benchmark column, and SPY
# must never be a candidate holding for the strategy itself.
scores = ranker.score_stocks(
    factor_table["momentum"], factor_table["low_volatility"], factor_table["dividend_yield"]
)
top_holdings = ranker.select_top_n(scores, n=config.TOP_N_HOLDINGS)

print(f"Top {config.TOP_N_HOLDINGS} holdings as of {as_of_date.date()}:")
print(list(top_holdings))

In [ ]:
window_start = as_of_date - pd.DateOffset(years=config.TRAIN_WINDOW_YEARS)
training_prices = prices.loc[
    (prices.index >= window_start) & (prices.index < as_of_date), list(top_holdings)
]
print("training window:", training_prices.index.min().date(), "to", training_prices.index.max().date())

weights = optimise.max_sharpe_weights(training_prices)
weights_series = pd.Series(weights).sort_values(ascending=False)
weights_series[weights_series > 0]

In [ ]:
nonzero_weights = weights_series[weights_series > 0]
nonzero_weights.plot(
    kind="bar", figsize=(10, 4), title=f"Max-Sharpe weights as of {as_of_date.date()}"
)
plt.axhline(config.MAX_WEIGHT_PER_STOCK, color="red", linestyle="--", label="per-stock cap")
plt.legend()
plt.ylabel("weight")
plt.show()